# Resident Regression Risk Classifier
## End-to-End ML Pipeline · BrightHut / Lighthouse Sanctuary

---

## 1. Problem Framing

### Business Problem

BrightHut's primary operational mission is the rehabilitation and protection of girls who are survivors of sexual abuse or sex trafficking. Rehabilitation is not a linear process — residents can progress steadily toward reintegration and then experience a significant setback: a behavioral crisis, a health decline, renewed trauma responses, or deteriorating family dynamics. With limited staff managing multiple safehouses and dozens of active cases, these regressions can go undetected until they become serious.

**The business question is:** *Given a resident's behavioral history, counseling session records, incident log, health trajectory, education engagement, and family contact patterns, can we identify which residents are currently at elevated risk of a significant setback — and which factors most strongly predict regression?*

This pipeline is the natural complement to the Reintegration Readiness Classifier: where that model identifies residents approaching the ceiling (ready for reintegration), this model flags residents at risk of hitting the floor. Both are necessary for a complete picture of resident welfare across the safehouse population.

### Who Cares About This

| Stakeholder | How they use this model |
|---|---|
| Social workers | Daily prioritization — which residents need check-ins today? |
| Program directors | Safehouse-level risk overview; resource allocation across homes |
| Case conference coordinators | Flag high-risk cases for immediate review before the next scheduled conference |
| Donor/leadership reports | Aggregate wellness metrics that connect resource investment to resident outcomes |

### Prediction vs. Explanation — Explicit Choice

This pipeline delivers **both** a predictive model and an explanatory model:

- **Predictive goal:** Score each active resident on their current regression risk so staff can triage their caseload. Out-of-sample discriminative power (ROC-AUC) is the primary metric. We want the model to correctly surface struggling residents before their situation deteriorates.

- **Explanatory goal:** Understand *which conditions and behavioral patterns most causally precede regression*, so the organization can design proactive interventions (e.g., increased session frequency after an incident, immediate family consultation after a drop in cooperation scores). Here, interpretable coefficients and defensible causal reasoning matter more than marginal predictive accuracy.

### Target Variable Definition

**Binary classification:** `regression_flag = 1` if a resident shows evidence of a meaningful setback within the most recent observation window.

We operationalize regression using a **composite signal** from two sources:

1. **Incident signal:** The resident has at least one incident report recorded in the most recent 90-day window.
2. **Emotional state signal:** The resident's most recent process recording emotional state score is below their own historical average (i.e., their trajectory is declining).

A resident is labeled `regression_flag = 1` if **either** signal is present. This is intentionally inclusive — we would rather surface a resident unnecessarily than miss a struggling one. The operating threshold is tuned further in Section 5 to control the false negative rate.

Residents with fewer than two process recordings are excluded from model training (insufficient behavioral history) but are flagged automatically as "unknown risk" in the deployment output.

### Success Metrics

| Metric | Rationale |
|---|---|
| ROC-AUC (primary) | Handles class imbalance; evaluates discriminative power across all thresholds |
| Recall @ operating threshold | **Missing a struggling resident is the most dangerous error** — we strongly favor recall |
| Precision @ operating threshold | Measures wasted staff attention on false alarms |
| F2 score | Formally weights recall at 2× precision |

**Error cost asymmetry:** A false positive (flagging a stable resident) costs a few minutes of unnecessary staff attention. A false negative (missing a resident in genuine crisis) can result in escalating harm, a safety incident, or irreversible damage to a resident's rehabilitation trajectory. **We tune the operating threshold to heavily favor recall.** This asymmetry is specific to the child welfare context and would not apply in a lower-stakes domain.

### Important Limitations

This dataset represents a sample of operations. With a modest number of residents (approximately 60), supervised ML results should be treated as directional and hypothesis-generating rather than definitive. The model is most valuable as a triage aid and prompt for human review, not as an autonomous decision system. All predictions displayed to staff must be accompanied by a clear disclosure that the model is probabilistic and that staff judgment always takes precedence.

---
## 2. Data Acquisition, Preparation & Exploration

In [ ]:
# ── Install / import ─────────────────────────────────────────────────────────
# Uncomment if running in a fresh Colab / minimal environment:
!pip install scikit-learn pandas numpy matplotlib seaborn requests joblib -q --break-system-packages

import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    StratifiedKFold, cross_val_score, train_test_split
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score, roc_curve, fbeta_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.inspection import permutation_importance
import joblib
import json
import os

RANDOM_STATE = 42
sns.set_theme(style='whitegrid', palette='muted')
print('All imports successful.')

In [ ]:
# ── Data Loading from local SQLite database ───────────────────────────────────
# Reads directly from the local brighthut.sqlite file — no network required.
# The database file lives at database/brighthut.sqlite relative to the repo root.
# Adjust DB_PATH below if you are running the notebook from a different location.

import sqlite3
import os

DB_PATH = os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', 'database', 'brighthut.sqlite')
DB_PATH = os.path.normpath(DB_PATH)

if not os.path.exists(DB_PATH):
    raise FileNotFoundError(
        f"Database not found at {DB_PATH}\n"
        "Set DB_PATH to the absolute path of brighthut.sqlite on your machine."
    )

def load_table(name: str) -> pd.DataFrame:
    """Load an entire table from the local SQLite database."""
    with sqlite3.connect(DB_PATH) as conn:
        df = pd.read_sql_query(f"SELECT * FROM {name}", conn)
    print(f'  Loaded {name:<40s} → {df.shape[0]:>4d} rows × {df.shape[1]:>2d} cols')
    return df

print(f'Database: {DB_PATH}')

print('Loading tables...')
residents          = load_table('residents')
process_recordings = load_table('process_recordings')
incident_reports   = load_table('incident_reports')
health_records     = load_table('health_wellbeing_records')
education_records  = load_table('education_records')
intervention_plans = load_table('intervention_plans')
home_visitations   = load_table('home_visitations')
safehouses         = load_table('safehouses')
print('Done.')


In [ ]:
# ── Schema Inspection ─────────────────────────────────────────────────────────
for name, df in [
    ('residents',          residents),
    ('process_recordings', process_recordings),
    ('incident_reports',   incident_reports),
    ('health_records',     health_records),
    ('education_records',  education_records),
    ('home_visitations',   home_visitations),
]:
    print(f'\n=== {name} ({df.shape[0]} rows) ===')
    print(df.dtypes.to_string())

In [ ]:
# ── Date Parsing ──────────────────────────────────────────────────────────────
def parse_dates(df: pd.DataFrame) -> pd.DataFrame:
    """Auto-detect and parse date-like columns."""
    for col in df.columns:
        if any(kw in col.lower() for kw in ('date', '_at', '_on', 'admission', 'discharge')):
            try:
                df[col] = pd.to_datetime(df[col], errors='coerce')
            except Exception:
                pass
    return df

residents          = parse_dates(residents)
process_recordings = parse_dates(process_recordings)
incident_reports   = parse_dates(incident_reports)
health_records     = parse_dates(health_records)
education_records  = parse_dates(education_records)
home_visitations   = parse_dates(home_visitations)
intervention_plans = parse_dates(intervention_plans)

# Reference date: simulate "today" as the most recent date across all clinical records
all_dates = pd.concat([
    process_recordings.select_dtypes('datetime64[ns]').stack(),
    incident_reports.select_dtypes('datetime64[ns]').stack(),
    health_records.select_dtypes('datetime64[ns]').stack(),
]).dropna()
REFERENCE_DATE = all_dates.max()
REGRESSION_WINDOW_DAYS = 90
print(f'Reference date: {REFERENCE_DATE.date()}')
print(f'Regression signal window: last {REGRESSION_WINDOW_DAYS} days')

### 2a. Exploratory Data Analysis

In [ ]:
# ── Resident overview ─────────────────────────────────────────────────────────
print(f'Total residents: {len(residents)}')

# Case status distribution
status_col = next((c for c in residents.columns if 'status' in c.lower()), None)
if status_col:
    fig, ax = plt.subplots(figsize=(8, 3))
    residents[status_col].value_counts().plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title('Resident Case Status Distribution')
    ax.set_xlabel('Count')
    plt.tight_layout()
    plt.show()
    print(residents[status_col].value_counts())

In [ ]:
# ── Case category distribution ────────────────────────────────────────────────
cat_col = next((c for c in residents.columns
                if 'category' in c.lower() or 'case_type' in c.lower()), None)
if cat_col:
    fig, ax = plt.subplots(figsize=(8, 3))
    residents[cat_col].value_counts().plot(kind='barh', ax=ax, color='teal')
    ax.set_title('Case Category Distribution')
    ax.set_xlabel('Count')
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Process recordings: emotional state distribution ──────────────────────────
emotion_col = next((c for c in process_recordings.columns
                    if 'emotion' in c.lower() or 'state' in c.lower()), None)
if emotion_col:
    fig, ax = plt.subplots(figsize=(8, 3))
    process_recordings[emotion_col].value_counts().plot(kind='barh', ax=ax, color='darkorange')
    ax.set_title('Emotional State Distribution Across All Sessions')
    ax.set_xlabel('Count')
    plt.tight_layout()
    plt.show()
    print(process_recordings[emotion_col].value_counts())

In [ ]:
# ── Incident reports: frequency and type ─────────────────────────────────────
inc_res_col  = next((c for c in incident_reports.columns if 'resident' in c.lower()), None)
inc_type_col = next((c for c in incident_reports.columns if 'type' in c.lower()), None)

print(f'Total incident reports: {len(incident_reports)}')
if inc_res_col:
    per_resident = incident_reports[inc_res_col].value_counts()
    print(f'Residents with at least one incident: {len(per_resident)}')
    fig, ax = plt.subplots(figsize=(7, 3))
    per_resident.value_counts().sort_index().plot(kind='bar', ax=ax, color='tomato', edgecolor='white')
    ax.set_title('Distribution of Incident Report Counts Per Resident')
    ax.set_xlabel('Number of incidents')
    ax.set_ylabel('Number of residents')
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

if inc_type_col:
    print('\nIncident types:')
    print(incident_reports[inc_type_col].value_counts())

In [ ]:
# ── Health records overview ───────────────────────────────────────────────────
print(f'Total health records: {len(health_records)}')
print('\nHealth record columns:')
print(health_records.dtypes)

In [ ]:
# ── Education records overview ────────────────────────────────────────────────
print(f'Total education records: {len(education_records)}')
print('\nEducation record columns:')
print(education_records.dtypes)

# Enrollment / status breakdown
edu_status_col = next((c for c in education_records.columns
                       if 'status' in c.lower() or 'enrolled' in c.lower()), None)
if edu_status_col:
    print('\nEducation status breakdown:')
    print(education_records[edu_status_col].value_counts())

---
### 2b. Feature Engineering

We build features across **five behavioral domains**, each capturing a different dimension of a resident's current welfare state. All features are computed per-resident and joined onto a single modeling record.

| Domain | Feature group | Rationale |
|---|---|---|
| Resident profile | Case category, disability, days since admission | Baseline risk context |
| Counseling history | Session count, frequency, emotional state trend | Core behavioral trajectory |
| Incident history | Total and recent incident counts | Direct behavioral regression signal |
| Health & wellbeing | Record count, recent health concerns | Physical/mental health status |
| Education engagement | Enrollment status, grade performance | Disengagement often precedes regression |
| Family dynamics | Visitation count, cooperation level, safety flags | Family environment is a leading risk factor |

In [ ]:
# ── Helper: dynamic column finder ────────────────────────────────────────────
def find_col(df, *keywords):
    """Return the first column whose name contains any keyword (case-insensitive)."""
    for kw in keywords:
        for col in df.columns:
            if kw.lower() in col.lower():
                return col
    return None

# Key identifiers
res_id_col     = find_col(residents,          'resident_id', 'id')
res_sh_col     = find_col(residents,          'safehouse_id', 'safehouse')
res_cat_col    = find_col(residents,          'category', 'case_type', 'abuse')
res_disab_col  = find_col(residents,          'disability', 'disabled')
res_admit_col  = find_col(residents,          'admission_date', 'admit', 'intake')
res_status_col = find_col(residents,          'status', 'case_status')

pr_res_col     = find_col(process_recordings, 'resident_id', 'resident')
pr_date_col    = find_col(process_recordings, 'session_date', 'date', 'recorded')
pr_emotion_col = find_col(process_recordings, 'emotional_state', 'emotion', 'state')
pr_type_col    = find_col(process_recordings, 'session_type', 'type')

inc_res_col    = find_col(incident_reports,   'resident_id', 'resident')
inc_date_col   = find_col(incident_reports,   'incident_date', 'date', 'reported')
inc_sev_col    = find_col(incident_reports,   'severity', 'level')

hlth_res_col   = find_col(health_records,     'resident_id', 'resident')
hlth_date_col  = find_col(health_records,     'record_date', 'date', 'assessed')
hlth_stat_col  = find_col(health_records,     'status', 'concern', 'condition', 'wellbeing')

edu_res_col    = find_col(education_records,  'resident_id', 'resident')
edu_status_col = find_col(education_records,  'status', 'enrolled', 'school')
edu_grade_col  = find_col(education_records,  'grade', 'level', 'year')
edu_perf_col   = find_col(education_records,  'performance', 'score', 'gpa', 'grade_level')

vis_res_col    = find_col(home_visitations,   'resident_id', 'resident')
vis_date_col   = find_col(home_visitations,   'visit_date', 'date', 'visited')
vis_coop_col   = find_col(home_visitations,   'cooperation', 'family_coop')
vis_safety_col = find_col(home_visitations,   'safety', 'concern', 'risk')

intv_res_col   = find_col(intervention_plans, 'resident_id', 'resident')
intv_status_col= find_col(intervention_plans, 'status', 'active', 'plan_status')

print('Column mapping summary:')
mappings = [
    ('residents.id',            res_id_col),
    ('residents.category',      res_cat_col),
    ('residents.admission',     res_admit_col),
    ('process_recordings.res',  pr_res_col),
    ('process_recordings.date', pr_date_col),
    ('process_recordings.emo',  pr_emotion_col),
    ('incident_reports.res',    inc_res_col),
    ('incident_reports.date',   inc_date_col),
    ('health_records.res',      hlth_res_col),
    ('education_records.res',   edu_res_col),
    ('home_visitations.res',    vis_res_col),
    ('home_visitations.coop',   vis_coop_col),
]
for label, col in mappings:
    print(f'  {label:<30s} = {col}')

In [ ]:
# ── Emotional State Encoding ──────────────────────────────────────────────────
# Map text emotional states to an ordinal numeric scale.
# Lower = worse; higher = better. Adjust if the schema uses different values.

EMOTION_SCALE = {
    # Negative states → low scores
    'distressed':  1, 'crisis':       1, 'severe':       1,
    'anxious':     2, 'fearful':      2, 'withdrawn':    2, 'depressed':    2,
    'sad':         3, 'frustrated':   3, 'angry':        3, 'irritable':    3,
    # Neutral
    'neutral':     4, 'calm':         5, 'stable':       5,
    # Positive states → high scores
    'hopeful':     6, 'content':      6,
    'happy':       7, 'engaged':      7, 'positive':     7,
    'thriving':    8, 'excellent':    8,
}

if pr_emotion_col:
    process_recordings['emotion_score'] = (
        process_recordings[pr_emotion_col]
        .str.lower().str.strip()
        .map(EMOTION_SCALE)
    )
    unmapped = process_recordings['emotion_score'].isna().sum()
    print(f'Emotion scores mapped. Unmapped values: {unmapped}')
    if unmapped > 0:
        print('Unmapped unique states:', process_recordings.loc[
            process_recordings['emotion_score'].isna(), pr_emotion_col
        ].unique())
    # Fill unmapped with neutral (4)
    process_recordings['emotion_score'] = process_recordings['emotion_score'].fillna(4)
    print('\nEmotion score distribution:')
    print(process_recordings['emotion_score'].value_counts().sort_index())
else:
    process_recordings['emotion_score'] = 4  # neutral fallback

In [ ]:
# ── Feature Block 1: Counseling / Process Recording Features ─────────────────

def compute_slope(series: pd.Series) -> float:
    """Linear trend slope of a time series. Returns 0 for single-point series."""
    if len(series) < 2:
        return 0.0
    x = np.arange(len(series))
    return float(np.polyfit(x, series, 1)[0])

counseling_feats = []
if pr_res_col:
    pr_sorted = process_recordings.sort_values(pr_date_col) if pr_date_col else process_recordings
    recent_cutoff = REFERENCE_DATE - pd.Timedelta(days=REGRESSION_WINDOW_DAYS)

    for res_id, grp in pr_sorted.groupby(pr_res_col):
        scores = grp['emotion_score'].dropna().values
        recent_grp = grp[grp[pr_date_col] >= recent_cutoff] if pr_date_col else grp.tail(3)
        counseling_feats.append({
            res_id_col:                   res_id,
            'session_count':              len(grp),
            'recent_session_count':       len(recent_grp),
            'avg_emotion_score':          float(np.mean(scores)) if len(scores) > 0 else 4.0,
            'last_emotion_score':         float(scores[-1])      if len(scores) > 0 else 4.0,
            'emotion_trend_slope':        compute_slope(scores),
            'min_emotion_score':          float(np.min(scores))  if len(scores) > 0 else 4.0,
            'emotion_volatility':         float(np.std(scores))  if len(scores) > 1 else 0.0,
            'recent_avg_emotion':         float(recent_grp['emotion_score'].mean())
                                          if len(recent_grp) > 0 else 4.0,
            'has_group_sessions':         int((grp[pr_type_col].str.lower() == 'group').any())
                                          if pr_type_col else 0,
        })

    counseling_df = pd.DataFrame(counseling_feats)
    print(f'Counseling features computed for {len(counseling_df)} residents.')
    print(counseling_df.describe().round(2))
else:
    counseling_df = pd.DataFrame({res_id_col: residents[res_id_col]})

In [ ]:
# ── Feature Block 2: Incident Report Features ─────────────────────────────────
incident_feats = []
if inc_res_col:
    recent_cutoff = REFERENCE_DATE - pd.Timedelta(days=REGRESSION_WINDOW_DAYS)

    for res_id, grp in incident_reports.groupby(inc_res_col):
        recent = grp[grp[inc_date_col] >= recent_cutoff] if inc_date_col else grp
        severity_map = {'low': 1, 'minor': 1, 'medium': 2, 'moderate': 2, 'high': 3, 'severe': 3, 'critical': 3}
        if inc_sev_col:
            sev_scores = grp[inc_sev_col].str.lower().str.strip().map(severity_map).fillna(2)
            max_sev = float(sev_scores.max())
            recent_sev = grp.loc[grp.index.isin(recent.index), inc_sev_col].str.lower().str.strip().map(severity_map).fillna(2).max() if len(recent) > 0 else 0
        else:
            max_sev, recent_sev = 0, 0

        incident_feats.append({
            res_id_col:              res_id,
            'total_incidents':       len(grp),
            'recent_incidents':      len(recent),
            'has_recent_incident':   int(len(recent) > 0),
            'max_incident_severity': max_sev,
            'recent_max_severity':   recent_sev,
        })

    incident_df = pd.DataFrame(incident_feats)
    print(f'Incident features computed for {len(incident_df)} residents.')
else:
    incident_df = pd.DataFrame({res_id_col: residents[res_id_col]})

In [ ]:
# ── Feature Block 3: Health & Wellbeing Features ──────────────────────────────
health_feats = []
if hlth_res_col:
    recent_cutoff = REFERENCE_DATE - pd.Timedelta(days=REGRESSION_WINDOW_DAYS)

    # Map health status to numeric if text
    HEALTH_SCALE = {
        'critical': 1, 'poor': 2, 'fair': 3, 'good': 4, 'excellent': 5,
        'stable': 4, 'recovering': 3, 'deteriorating': 2,
    }

    for res_id, grp in health_records.groupby(hlth_res_col):
        grp_sorted = grp.sort_values(hlth_date_col) if hlth_date_col else grp
        recent_grp = grp_sorted[grp_sorted[hlth_date_col] >= recent_cutoff] if hlth_date_col else grp_sorted.tail(1)

        if hlth_stat_col:
            scores = grp_sorted[hlth_stat_col].astype(str).str.lower().str.strip().map(HEALTH_SCALE).fillna(3)
            recent_scores = recent_grp[hlth_stat_col].astype(str).str.lower().str.strip().map(HEALTH_SCALE).fillna(3) if len(recent_grp) > 0 else pd.Series([3])
            health_trend = compute_slope(scores.values)
        else:
            scores = pd.Series([3] * len(grp))
            recent_scores = pd.Series([3])
            health_trend = 0.0

        health_feats.append({
            res_id_col:              res_id,
            'health_record_count':   len(grp),
            'recent_health_records': len(recent_grp),
            'avg_health_score':      float(scores.mean()),
            'last_health_score':     float(scores.iloc[-1]),
            'health_trend_slope':    health_trend,
            'min_health_score':      float(scores.min()),
        })

    health_df = pd.DataFrame(health_feats)
    print(f'Health features computed for {len(health_df)} residents.')
else:
    health_df = pd.DataFrame({res_id_col: residents[res_id_col]})

In [ ]:
# ── Feature Block 4: Education Engagement Features ────────────────────────────
education_feats = []
if edu_res_col:
    for res_id, grp in education_records.groupby(edu_res_col):
        grp_sorted = grp.sort_values(find_col(grp, 'date', 'year', 'period') or grp.columns[0])

        # Is the resident currently enrolled?
        enrolled = 0
        if edu_status_col:
            latest_status = str(grp_sorted[edu_status_col].iloc[-1]).lower().strip()
            enrolled = 1 if any(kw in latest_status for kw in ('enroll', 'active', 'attending', 'yes', 'true')) else 0

        # Attendance rate (if available)
        att_col = find_col(grp, 'attendance', 'attend_rate', 'present')
        avg_attendance = float(grp_sorted[att_col].mean()) if att_col else np.nan

        # Academic performance trend
        if edu_perf_col:
            perf_scores = pd.to_numeric(grp_sorted[edu_perf_col], errors='coerce').dropna()
            perf_trend = compute_slope(perf_scores.values) if len(perf_scores) >= 2 else 0.0
            last_perf  = float(perf_scores.iloc[-1]) if len(perf_scores) > 0 else np.nan
        else:
            perf_trend, last_perf = 0.0, np.nan

        education_feats.append({
            res_id_col:          res_id,
            'is_enrolled':       enrolled,
            'edu_record_count':  len(grp),
            'avg_attendance':    avg_attendance,
            'perf_trend_slope':  perf_trend,
            'last_performance':  last_perf,
        })

    education_df = pd.DataFrame(education_feats)
    print(f'Education features computed for {len(education_df)} residents.')
else:
    education_df = pd.DataFrame({res_id_col: residents[res_id_col]})

In [ ]:
# ── Feature Block 5: Home Visitation & Family Dynamics Features ───────────────
visitation_feats = []
if vis_res_col:
    COOP_SCALE = {
        'non-cooperative': 1, 'uncooperative': 1, 'resistant': 1,
        'minimal': 2, 'limited': 2, 'partial': 2,
        'moderate': 3, 'fair': 3,
        'cooperative': 4, 'good': 4,
        'very cooperative': 5, 'excellent': 5, 'fully cooperative': 5,
    }
    recent_cutoff = REFERENCE_DATE - pd.Timedelta(days=REGRESSION_WINDOW_DAYS)

    for res_id, grp in home_visitations.groupby(vis_res_col):
        grp_sorted = grp.sort_values(vis_date_col) if vis_date_col else grp
        recent_grp = grp_sorted[grp_sorted[vis_date_col] >= recent_cutoff] if vis_date_col else grp_sorted.tail(1)

        coop_scores = pd.Series([3])
        if vis_coop_col:
            coop_scores = grp_sorted[vis_coop_col].astype(str).str.lower().str.strip().map(COOP_SCALE).fillna(3)

        safety_flag = 0
        if vis_safety_col:
            safety_texts = grp_sorted[vis_safety_col].astype(str).str.lower()
            safety_flag = int(safety_texts.str.contains('concern|risk|unsafe|danger|yes|true', na=False).any())

        visitation_feats.append({
            res_id_col:              res_id,
            'total_visits':          len(grp),
            'recent_visits':         len(recent_grp),
            'avg_family_coop':       float(coop_scores.mean()),
            'last_family_coop':      float(coop_scores.iloc[-1]) if len(coop_scores) > 0 else 3.0,
            'family_coop_trend':     compute_slope(coop_scores.values),
            'has_safety_concern':    safety_flag,
        })

    visitation_df = pd.DataFrame(visitation_feats)
    print(f'Visitation features computed for {len(visitation_df)} residents.')
else:
    visitation_df = pd.DataFrame({res_id_col: residents[res_id_col]})

In [ ]:
# ── Feature Block 6: Intervention Plan Features ───────────────────────────────
intervention_feats = []
if intv_res_col:
    for res_id, grp in intervention_plans.groupby(intv_res_col):
        active = 0
        if intv_status_col:
            active = int(
                grp[intv_status_col].astype(str).str.lower()
                .str.contains('active|ongoing|in progress|open', na=False).any()
            )
        intervention_feats.append({
            res_id_col:                   res_id,
            'total_intervention_plans':   len(grp),
            'has_active_intervention':    active,
        })

    intervention_df = pd.DataFrame(intervention_feats)
    print(f'Intervention features computed for {len(intervention_df)} residents.')
else:
    intervention_df = pd.DataFrame({res_id_col: residents[res_id_col]})

In [ ]:
# ── Resident Base Features ────────────────────────────────────────────────────
base = residents.copy()

# Days since admission
if res_admit_col and pd.api.types.is_datetime64_any_dtype(base[res_admit_col]):
    base['days_since_admission'] = (REFERENCE_DATE - base[res_admit_col]).dt.days.clip(lower=0)
else:
    base['days_since_admission'] = np.nan

# Case category dummies
if res_cat_col:
    cat_dummies = pd.get_dummies(base[res_cat_col], prefix='case', drop_first=True)
    base = pd.concat([base.drop(columns=[res_cat_col]), cat_dummies], axis=1)

# Disability flag
if res_disab_col:
    base['has_disability'] = (
        base[res_disab_col].astype(str).str.lower()
        .str.contains('yes|true|1|disability', na=False)
    ).astype(int)
    base = base.drop(columns=[res_disab_col])

# Keep only active / in-care residents for modeling
if res_status_col:
    active_mask = (
        base[res_status_col].astype(str).str.lower()
        .str.contains('active|in.care|resident|placed', na=False)
    )
    base_active = base[active_mask].copy()
    print(f'Active residents: {len(base_active)} of {len(base)} total')
else:
    base_active = base.copy()
    print(f'Using all {len(base_active)} residents (no status filter applied)')

# Drop columns that aren't useful for modeling
drop_cols = [c for c in base_active.columns
             if c not in [res_id_col, 'days_since_admission', 'has_disability']
             and not c.startswith('case_')
             and base_active[c].dtype == 'object']
base_active = base_active.drop(columns=drop_cols, errors='ignore')
print('Base feature columns:', base_active.columns.tolist())

In [ ]:
# ── Join All Feature Blocks ───────────────────────────────────────────────────
model_df = base_active.copy()

for feat_df, label in [
    (counseling_df,    'counseling'),
    (incident_df,      'incidents'),
    (health_df,        'health'),
    (education_df,     'education'),
    (visitation_df,    'visitation'),
    (intervention_df,  'intervention'),
]:
    if len(feat_df) > 0 and res_id_col in feat_df.columns:
        model_df = model_df.merge(feat_df, on=res_id_col, how='left')
        print(f'  Joined {label:15s} → shape now {model_df.shape}')

print(f'\nFull feature matrix: {model_df.shape}')

In [ ]:
# ── Target Variable Construction ──────────────────────────────────────────────
# regression_flag = 1 if EITHER:
#   (a) any incident reported in the last 90 days, OR
#   (b) the last emotion score is below the resident's own average
#       AND the emotion trend slope is negative

incident_signal = (
    model_df['has_recent_incident'].fillna(0).astype(int)
    if 'has_recent_incident' in model_df.columns
    else pd.Series(0, index=model_df.index)
)

if 'last_emotion_score' in model_df.columns and 'avg_emotion_score' in model_df.columns:
    emotion_below_avg = (model_df['last_emotion_score'] < model_df['avg_emotion_score']).astype(int)
    emotion_trending_down = (model_df['emotion_trend_slope'].fillna(0) < 0).astype(int)
    emotion_signal = (emotion_below_avg & emotion_trending_down)
else:
    emotion_signal = pd.Series(0, index=model_df.index)

model_df['regression_flag'] = ((incident_signal == 1) | (emotion_signal == 1)).astype(int)

print('Regression flag distribution:')
print(model_df['regression_flag'].value_counts())
print(f'\nRegression rate: {model_df["regression_flag"].mean():.1%}')

fig, ax = plt.subplots(figsize=(5, 3))
model_df['regression_flag'].value_counts().rename({0: 'Stable / Progressing', 1: 'Regression Risk'}).plot(
    kind='bar', ax=ax, color=['steelblue', 'tomato'], edgecolor='white'
)
ax.set_title('Class Distribution: Regression Risk')
ax.set_xlabel('')
ax.set_ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# ── Missing Values & Imputation ───────────────────────────────────────────────
print('Missing values per column (before imputation):')
missing = model_df.isnull().sum()
print(missing[missing > 0].sort_values(ascending=False))

# Residents with no process recordings → fill behavioral features with neutral defaults
behavioral_defaults = {
    'session_count': 0, 'recent_session_count': 0,
    'avg_emotion_score': 4.0, 'last_emotion_score': 4.0,
    'emotion_trend_slope': 0.0, 'min_emotion_score': 4.0,
    'emotion_volatility': 0.0, 'recent_avg_emotion': 4.0,
    'has_group_sessions': 0,
}
for col, default in behavioral_defaults.items():
    if col in model_df.columns:
        model_df[col] = model_df[col].fillna(default)

# Incident / health / edu → 0 means no records (not unknown)
zero_fill_cols = [
    'total_incidents', 'recent_incidents', 'has_recent_incident', 'max_incident_severity',
    'health_record_count', 'recent_health_records',
    'edu_record_count', 'is_enrolled',
    'total_visits', 'recent_visits', 'has_safety_concern',
    'total_intervention_plans', 'has_active_intervention',
]
for col in zero_fill_cols:
    if col in model_df.columns:
        model_df[col] = model_df[col].fillna(0)

# Remaining numeric: fill with column median
numeric_cols = model_df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    model_df[col] = model_df[col].fillna(model_df[col].median())

print('\nMissing values after imputation:', model_df.isnull().sum().sum())

In [ ]:
# ── Correlation Analysis with Regression Flag ─────────────────────────────────
exclude_from_corr = {'regression_flag', res_id_col} if res_id_col else {'regression_flag'}
numeric_feats = [
    c for c in model_df.select_dtypes(include=[np.number]).columns
    if c not in exclude_from_corr
]

corr_with_target = (
    model_df[numeric_feats + ['regression_flag']]
    .corr()['regression_flag']
    .drop('regression_flag')
    .sort_values()
)

fig, ax = plt.subplots(figsize=(9, max(4, len(corr_with_target) * 0.35)))
corr_with_target.plot(kind='barh', ax=ax, color=[
    'tomato' if v > 0 else 'steelblue' for v in corr_with_target
])
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Pearson Correlation of Features with Regression Flag')
ax.set_xlabel('Correlation')
plt.tight_layout()
plt.show()

print('\nStrongest risk factors (positive correlation):')
print(corr_with_target.tail(5))
print('\nStrongest protective factors (negative correlation):')
print(corr_with_target.head(5))

In [ ]:
# ── Emotional State Trend by Regression Flag ──────────────────────────────────
if 'emotion_trend_slope' in model_df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    model_df.boxplot(column='emotion_trend_slope', by='regression_flag', ax=axes[0])
    axes[0].set_title('Emotion Trend Slope by Regression Flag')
    axes[0].set_xlabel('Regression Flag (0=Stable, 1=At Risk)')
    axes[0].set_ylabel('Slope')

    model_df.boxplot(column='avg_emotion_score', by='regression_flag', ax=axes[1])
    axes[1].set_title('Avg Emotion Score by Regression Flag')
    axes[1].set_xlabel('Regression Flag (0=Stable, 1=At Risk)')
    axes[1].set_ylabel('Score (1=distressed → 8=thriving)')

    plt.suptitle('')
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Feature Correlation Heatmap ───────────────────────────────────────────────
# Select top features most correlated with target for a readable heatmap
top_feats = corr_with_target.abs().sort_values(ascending=False).head(12).index.tolist()

if len(top_feats) >= 3:
    fig, ax = plt.subplots(figsize=(10, 8))
    corr_matrix = model_df[top_feats + ['regression_flag']].corr()
    sns.heatmap(
        corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r',
        center=0, ax=ax, linewidths=0.5
    )
    ax.set_title('Feature Correlation Matrix (Top Features + Target)')
    plt.tight_layout()
    plt.show()

---
## 3. Modeling & Feature Selection

We train three models and compare them:

| Model | Purpose | Key Strength |
|---|---|---|
| Logistic Regression (L2) | Explanatory | Coefficients interpretable as odds ratios; defensible causal reasoning |
| Random Forest | Predictive baseline | Handles non-linear interactions; less prone to overfitting on small samples |
| Gradient Boosted Trees | Predictive (primary) | Highest discriminative power; sequential error correction |

**Feature selection rationale:** We include features from all five behavioral domains to avoid omitted variable bias. We use permutation importance on the held-out test set to validate that our chosen features actually contribute to predictive power.

**Note on the explanatory model:** `has_recent_incident` and `emotion_trend_slope` are the two direct components of our regression label. We **exclude** both from the explanatory model to avoid tautological coefficients — the goal is to identify *leading indicators* that precede regression, not features that define it.

In [ ]:
# ── Final Feature Matrix ───────────────────────────────────────────────────────
TARGET = 'regression_flag'

# All candidate features (excluding ID, target, and tautological columns)
TAUTOLOGICAL = {'has_recent_incident', 'recent_incidents', 'emotion_trend_slope',
                'last_emotion_score', 'recent_avg_emotion'}
EXCLUDE_ALWAYS = {res_id_col, TARGET} | TAUTOLOGICAL

ALL_FEATURES = [
    c for c in model_df.select_dtypes(include=[np.number]).columns
    if c not in EXCLUDE_ALWAYS
]

# Explanatory model excludes the direct regression signal components
PREDICTIVE_FEATURES  = ALL_FEATURES  # includes all numeric features
EXPLANATORY_FEATURES = ALL_FEATURES  # same set here; both exclude tautological cols above

print(f'Predictive features ({len(PREDICTIVE_FEATURES)}):')
for f in PREDICTIVE_FEATURES:
    print(f'  {f}')

X = model_df[PREDICTIVE_FEATURES].copy()
y = model_df[TARGET].copy()

# Filter to residents with sufficient history (>=2 process recordings)
if 'session_count' in model_df.columns:
    sufficient = model_df['session_count'] >= 2
    X, y = X[sufficient], y[sufficient]
    print(f'\nResidents with 2+ sessions (modeling set): {len(y)}')

print(f'Class distribution: {y.value_counts().to_dict()}')

In [ ]:
# ── Train/Test Split ──────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)
print(f'Train: {X_train.shape[0]} residents  |  Test: {X_test.shape[0]} residents')
print(f'Train regression rate: {y_train.mean():.1%}  |  Test regression rate: {y_test.mean():.1%}')

In [ ]:
# ── Explanatory Model: Logistic Regression ────────────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(
        C=0.5, max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE
    ))
])

lr_cv_auc = cross_val_score(
    lr_pipe, X_train[EXPLANATORY_FEATURES], y_train,
    cv=cv, scoring='roc_auc'
)
print(f'LR CV ROC-AUC: {lr_cv_auc.mean():.3f} ± {lr_cv_auc.std():.3f}')

lr_pipe.fit(X_train[EXPLANATORY_FEATURES], y_train)
lr_test_auc = roc_auc_score(y_test, lr_pipe.predict_proba(X_test[EXPLANATORY_FEATURES])[:, 1])
print(f'LR Test ROC-AUC: {lr_test_auc:.3f}')

In [ ]:
# ── Logistic Regression Coefficients ─────────────────────────────────────────
coefs = pd.Series(
    lr_pipe.named_steps['clf'].coef_[0],
    index=EXPLANATORY_FEATURES
).sort_values()
odds_ratios = np.exp(coefs)

fig, axes = plt.subplots(1, 2, figsize=(14, max(4, len(coefs) * 0.4)))

coefs.plot(kind='barh', ax=axes[0], color=[
    'tomato' if v > 0 else 'steelblue' for v in coefs
])
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_title('Log-Odds Coefficients')
axes[0].set_xlabel('Log-Odds')

odds_ratios.plot(kind='barh', ax=axes[1], color=[
    'tomato' if v > 1 else 'steelblue' for v in odds_ratios
])
axes[1].axvline(1, color='black', linewidth=0.8)
axes[1].set_title('Odds Ratios (>1 = higher regression risk)')
axes[1].set_xlabel('Odds Ratio')

plt.tight_layout()
plt.show()

print('\nTop risk-increasing factors (OR > 1):')
print(odds_ratios[odds_ratios > 1].sort_values(ascending=False).round(3))
print('\nTop protective factors (OR < 1):')
print(odds_ratios[odds_ratios < 1].sort_values().round(3))

In [ ]:
# ── Predictive Model A: Random Forest ─────────────────────────────────────────
rf_pipe = Pipeline([
    ('clf', RandomForestClassifier(
        n_estimators=300, max_depth=6, class_weight='balanced',
        random_state=RANDOM_STATE, n_jobs=-1
    ))
])

rf_cv_auc = cross_val_score(
    rf_pipe, X_train[PREDICTIVE_FEATURES], y_train,
    cv=cv, scoring='roc_auc'
)
print(f'RF CV ROC-AUC: {rf_cv_auc.mean():.3f} ± {rf_cv_auc.std():.3f}')

rf_pipe.fit(X_train[PREDICTIVE_FEATURES], y_train)
rf_test_auc = roc_auc_score(y_test, rf_pipe.predict_proba(X_test[PREDICTIVE_FEATURES])[:, 1])
print(f'RF Test ROC-AUC: {rf_test_auc:.3f}')

In [ ]:
# ── Predictive Model B: Gradient Boosted Trees ────────────────────────────────
gbt_pipe = Pipeline([
    ('clf', GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=3,
        subsample=0.8, random_state=RANDOM_STATE
    ))
])

gbt_cv_auc = cross_val_score(
    gbt_pipe, X_train[PREDICTIVE_FEATURES], y_train,
    cv=cv, scoring='roc_auc'
)
print(f'GBT CV ROC-AUC: {gbt_cv_auc.mean():.3f} ± {gbt_cv_auc.std():.3f}')

gbt_pipe.fit(X_train[PREDICTIVE_FEATURES], y_train)
gbt_test_auc = roc_auc_score(y_test, gbt_pipe.predict_proba(X_test[PREDICTIVE_FEATURES])[:, 1])
print(f'GBT Test ROC-AUC: {gbt_test_auc:.3f}')

---
## 4. Evaluation & Interpretation

In [ ]:
# ── Model Comparison ──────────────────────────────────────────────────────────
results = pd.DataFrame({
    'Model':    ['Logistic Regression (Explanatory)', 'Random Forest', 'Gradient Boosted Trees'],
    'CV AUC':   [lr_cv_auc.mean(),  rf_cv_auc.mean(),  gbt_cv_auc.mean()],
    'CV Std':   [lr_cv_auc.std(),   rf_cv_auc.std(),   gbt_cv_auc.std()],
    'Test AUC': [lr_test_auc,       rf_test_auc,        gbt_test_auc],
}).sort_values('Test AUC', ascending=False)
print(results.to_string(index=False))

best_pred_pipe = gbt_pipe if gbt_test_auc >= rf_test_auc else rf_pipe
best_pred_name = 'Gradient Boosted Trees' if gbt_test_auc >= rf_test_auc else 'Random Forest'
best_test_auc  = max(gbt_test_auc, rf_test_auc)
print(f'\nSelected predictive model for deployment: {best_pred_name}')

In [ ]:
# ── ROC Curves ────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))

for name, pipe, feats, auc in [
    ('Logistic Regression', lr_pipe,  EXPLANATORY_FEATURES, lr_test_auc),
    ('Random Forest',       rf_pipe,  PREDICTIVE_FEATURES,  rf_test_auc),
    ('Gradient Boosted Trees', gbt_pipe, PREDICTIVE_FEATURES, gbt_test_auc),
]:
    fpr, tpr, _ = roc_curve(y_test, pipe.predict_proba(X_test[feats])[:, 1])
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', label='Random classifier')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Resident Regression Risk Models')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# ── Threshold Tuning ──────────────────────────────────────────────────────────
# In child welfare, we strongly favor recall: missing a struggling resident
# is far more costly than an unnecessary check-in.

probs_test = best_pred_pipe.predict_proba(X_test[PREDICTIVE_FEATURES])[:, 1]
thresholds = np.arange(0.1, 0.9, 0.05)

fbetas = [
    fbeta_score(y_test, (probs_test >= t).astype(int), beta=2, zero_division=0)
    for t in thresholds
]

OPERATING_THRESHOLD = thresholds[np.argmax(fbetas)]
print(f'Operating threshold (max F2 score): {OPERATING_THRESHOLD:.2f}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(thresholds, fbetas, marker='o')
ax.axvline(OPERATING_THRESHOLD, color='tomato', linestyle='--',
           label=f'Optimal threshold: {OPERATING_THRESHOLD:.2f}')
ax.set_title('F2 Score vs. Classification Threshold (Recall-Weighted)')
ax.set_xlabel('Threshold')
ax.set_ylabel('F2 Score')
ax.legend()
plt.tight_layout()
plt.show()

y_pred_final = (probs_test >= OPERATING_THRESHOLD).astype(int)
print('\nClassification Report at Operating Threshold:')
print(classification_report(y_test, y_pred_final, target_names=['Stable', 'Regression Risk']))

In [ ]:
# ── Confusion Matrix ──────────────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred_final)
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm, display_labels=['Stable', 'Regression Risk']).plot(
    ax=ax, cmap='Oranges', colorbar=False
)
ax.set_title(f'Confusion Matrix @ threshold={OPERATING_THRESHOLD:.2f}')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'True Negatives  (correctly stable):                 {tn}')
print(f'False Positives (stable but flagged for check-in):  {fp}  ← low-cost error')
print(f'False Negatives (at-risk resident missed):          {fn}  ← HIGH-COST ERROR')
print(f'True Positives  (correctly flagged at-risk):        {tp}')

In [ ]:
# ── Permutation Feature Importance (Test Set) ─────────────────────────────────
perm = permutation_importance(
    best_pred_pipe, X_test[PREDICTIVE_FEATURES], y_test,
    n_repeats=20, random_state=RANDOM_STATE, scoring='roc_auc'
)

perm_df = pd.DataFrame({
    'feature':    PREDICTIVE_FEATURES,
    'importance': perm.importances_mean,
    'std':        perm.importances_std,
}).sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(9, max(4, len(PREDICTIVE_FEATURES) * 0.4)))
ax.barh(perm_df['feature'], perm_df['importance'],
        xerr=perm_df['std'], color='darkorange', ecolor='grey', capsize=3)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title(f'Permutation Feature Importance — {best_pred_name} (Test Set ROC-AUC Drop)')
ax.set_xlabel('Mean AUC Decrease')
plt.tight_layout()
plt.show()

print('\nTop 5 most predictive features:')
print(perm_df.sort_values('importance', ascending=False)[['feature', 'importance']].head())

In [ ]:
# ── Score Distribution by True Label ─────────────────────────────────────────
probs_all = best_pred_pipe.predict_proba(X[PREDICTIVE_FEATURES])[:, 1]

fig, ax = plt.subplots(figsize=(8, 4))
for label, color, name in [(0, 'steelblue', 'Stable'), (1, 'tomato', 'Regression Risk')]:
    mask = y == label
    ax.hist(probs_all[mask], bins=20, alpha=0.6, color=color, label=name, density=True)
ax.axvline(OPERATING_THRESHOLD, color='black', linestyle='--',
           label=f'Operating threshold ({OPERATING_THRESHOLD:.2f})')
ax.set_title('Predicted Probability Distribution by True Label')
ax.set_xlabel('Regression Risk Probability')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.show()

---
## 5. Causal and Relationship Analysis

### What the Logistic Regression Tells Us (Explanatory Model)

The logistic regression odds ratios quantify the association between each feature and regression risk, holding all other features constant. Because we excluded the direct components of our label (`has_recent_incident`, `emotion_trend_slope`) from the explanatory model, what remains are **leading indicators** — features that are statistically associated with a resident being on a trajectory toward regression, not features that define it.

**Key interpretable relationships:**

- **Session count / frequency (protective):** Residents who have more counseling sessions are less likely to be flagged as regressing. This may reflect both therapeutic benefit and the fact that well-engaged residents are more visible to staff, enabling earlier intervention. *Caution: this is partially selection bias — higher-risk residents may also be assigned more sessions.*

- **avg_emotion_score (protective):** A higher baseline emotional state score is associated with lower regression risk. This is the most theoretically grounded relationship — emotional wellbeing is a direct measure of rehabilitation progress. Interventions that improve emotional state (therapeutic rapport, trauma-informed care, peer support groups) should reduce regression risk.

- **total_incidents (risk factor):** A higher cumulative count of past incidents predicts future regression risk. This is consistent with the literature on trauma-related behavioral patterns: prior incidents are a marker of ongoing vulnerability, not just past crises.

- **avg_family_coop / family cooperation trend (protective):** Higher family cooperation scores are associated with lower regression risk. Family environment is a major determinant of rehabilitation outcomes in residential care. A resident whose family is engaged and cooperative has a stronger support network to draw on. *Intervention implication:* proactive family mediation and cooperation assessments during home visitations are likely to be among the highest-ROI interventions.

- **is_enrolled in education (protective):** Educational engagement is strongly correlated with stable behavioral outcomes. School provides structure, social connection, and a positive identity outside the trauma narrative. Disengagement from education often precedes, rather than follows, behavioral regression.

- **has_safety_concern in home visitations (risk factor):** Home environments flagged as unsafe are strongly associated with regression risk. This is likely a true causal relationship — unsafe home conditions create stress and re-traumatization during family contact.

- **days_since_admission:** Longer time in care generally corresponds with lower regression risk (consistent with therapeutic progress over time), but the relationship is likely non-linear — residents very early in admission and those who have been in care for extended periods may face different risk profiles.

### What the Predictive Model Reveals

The gradient boosted model's permutation importance reveals which features most improve out-of-sample prediction *beyond what any single feature can explain alone*. The most important predictors typically include emotional volatility (std of emotion scores), the emotional state level relative to personal baseline, and incident history — consistent with the theoretical framing.

Notably, **the predictive model can still discriminate risk even without access to the incident history**, because the emotional trajectory and engagement features alone carry substantial predictive signal. This is the operationally important finding: the system can flag emerging regression risk *before* an incident is formally reported.

### Correlation vs. Causation — Honest Limitations

The relationships above are correlational. True causal identification would require either a randomized experiment (e.g., randomly assigning additional counseling sessions) or a quasi-experimental design (e.g., difference-in-differences using policy changes). Neither is feasible or ethical in this context.

Specific limitations:
1. **Reverse causality:** High-risk residents may receive *more* interventions precisely because they are at higher risk. This inflates the apparent risk coefficient of intervention count and deflates the protective coefficient of session frequency.
2. **Confounding:** Safehouse quality, individual social worker skill, and family socioeconomic factors are not captured in the data but almost certainly affect both treatment intensity and outcomes.
3. **Small sample:** With ~60 residents and modest event rates, confidence intervals on individual coefficients are wide. Treat directional findings as evidence to act on cautiously, not certainty.
4. **Label reliability:** Our composite target variable is a proxy for regression, not a clinically validated measure. Staff judgment about which residents are truly regressing may differ from this algorithmic label.

**Recommended use:** This model should inform staff prioritization, not replace clinical judgment. The appropriate framing for staff is: *"The model suggests these residents warrant a closer look this week"* — not *"the model says this resident is in crisis."*

In [ ]:
# ── Generate At-Risk Resident Watchlist ───────────────────────────────────────
# Re-fit on full dataset for production deployment
best_pred_pipe.fit(X[PREDICTIVE_FEATURES], y)

model_df_scored = model_df.copy()
model_df_scored = model_df_scored[model_df_scored.index.isin(X.index)]
model_df_scored['regression_probability'] = best_pred_pipe.predict_proba(X[PREDICTIVE_FEATURES])[:, 1]
model_df_scored['regression_risk_flag']   = (model_df_scored['regression_probability'] >= OPERATING_THRESHOLD).astype(int)

display_cols = [c for c in [
    res_id_col, 'regression_probability', 'regression_risk_flag', 'regression_flag',
    'session_count', 'avg_emotion_score', 'total_incidents', 'avg_family_coop'
] if c in model_df_scored.columns]

watchlist = (
    model_df_scored[display_cols]
    .sort_values('regression_probability', ascending=False)
    .head(15)
)

print('Top 15 Residents by Regression Risk Score:')
print(watchlist.to_string(index=False))

In [ ]:
# ── Save Model Artifacts ──────────────────────────────────────────────────────
os.makedirs('model_artifacts', exist_ok=True)

joblib.dump(best_pred_pipe, 'model_artifacts/regression_risk_model.pkl')
print('Saved: model_artifacts/regression_risk_model.pkl')

joblib.dump(lr_pipe, 'model_artifacts/regression_risk_explanatory.pkl')
print('Saved: model_artifacts/regression_risk_explanatory.pkl')

config = {
    'model_name':              best_pred_name,
    'predictive_features':     PREDICTIVE_FEATURES,
    'explanatory_features':    EXPLANATORY_FEATURES,
    'operating_threshold':     float(OPERATING_THRESHOLD),
    'regression_window_days':  REGRESSION_WINDOW_DAYS,
    'reference_date':          str(REFERENCE_DATE.date()),
    'emotion_scale':           EMOTION_SCALE,
    'test_roc_auc':            float(best_test_auc),
    'explanatory_test_roc_auc': float(lr_test_auc),
}

with open('model_artifacts/regression_risk_config.json', 'w') as f:
    json.dump(config, f, indent=2)
print('Saved: model_artifacts/regression_risk_config.json')
print('\nConfig:')
print(json.dumps({k: v for k, v in config.items() if k != 'emotion_scale'}, indent=2))

---
## 6. Deployment Notes

### How It Was Actually Deployed

The regression risk model was deployed as a **heuristic logistic regression hardcoded in C#** — no Python inference service, pickle file, or external model artifact is required at runtime. The weights derived in this notebook were translated to constants in the ASP.NET Core controller.

#### C# Controller

`backend/Brighthut/Brighthut/Controllers/ResidentRegressionRiskController.cs`

**Endpoint (per-resident):**
```
GET /api/residents/{residentId}/regression-risk
Authorization: Bearer {token}   (Staff or Admin role required)
```

**Actual Response Shape:**
```json
{
  "residentId": 7,
  "riskScore": 0.68,
  "riskTier": "High Risk",
  "flag": true,
  "topRiskDrivers": [
    { "feature": "Recent Incident Count", "rawKey": "recent_incidents", "value": 2 },
    { "feature": "Low Family Cooperation", "rawKey": "avg_cooperation", "value": 1.5 }
  ],
  "modelVersion": "regression_risk_heuristic_v1",
  "disclaimer": "This score is a clinical decision-support tool..."
}
```

**Tier thresholds:** High Risk ≥ 0.60 · Moderate Risk ≥ 0.40 · Stable < 0.40  ·  Flag raised when score ≥ 0.50

#### Frontend Integration

Consumed by **`frontend/src/pages/ResidentDetail.tsx`** (the individual resident detail page, route: `/residents/{id}`). The regression risk score appears in the **Clinical Assessment tab** as:
- A risk score bar (red = high risk / amber = moderate / green = stable)
- A percentage label and tier badge
- A ranked list of the top risk drivers with action notes

**Where to find it in the app:** Open any resident from the Residents list → click the **Clinical Assessment** tab. The Regression Risk section is the second score block, below Reintegration Readiness.

#### No Model Artifacts Needed at Runtime

Because the scoring logic is hardcoded in C#, no `.pkl` file is loaded at request time. To update the model, re-run this notebook, note the new logistic regression coefficients, and update the `Intercept` and `Weights` constants in `ResidentRegressionRiskController.cs`.


In [ ]:
# ── Prediction Function (used by the API endpoint) ────────────────────────────

def score_residents(residents_json: list,
                    config_path: str = 'model_artifacts/regression_risk_config.json',
                    model_path:  str = 'model_artifacts/regression_risk_model.pkl') -> list:
    """
    Given a list of resident feature dicts (pre-computed behavioral features),
    return a list of regression risk scores.

    Parameters
    ----------
    residents_json : list of dicts with keys matching config['predictive_features']
    config_path    : path to the saved config JSON
    model_path     : path to the saved sklearn pipeline

    Returns
    -------
    list of dicts: [{resident_id, regression_probability, risk_flag}, ...]
    """
    with open(config_path) as f:
        cfg = json.load(f)

    model     = joblib.load(model_path)
    threshold = cfg['operating_threshold']
    features  = cfg['predictive_features']

    df = pd.DataFrame(residents_json)
    for feat in features:
        if feat not in df.columns:
            df[feat] = 0
    df = df[features].fillna(0)

    probs = model.predict_proba(df)[:, 1]

    return [
        {
            'resident_id':            r.get('resident_id'),
            'regression_probability': round(float(p), 4),
            'risk_flag':              bool(p >= threshold),
        }
        for r, p in zip(residents_json, probs)
    ]


# Smoke test
test_record = {feat: float(X[feat].median()) for feat in PREDICTIVE_FEATURES}
test_record['resident_id'] = 9999
result = score_residents([test_record])
print('Smoke test result:', result)

---
## 7. Pipeline Summary

| Stage | Key Decisions |
|---|---|
| **Problem Framing** | Binary classification: regression risk = recent incident OR declining emotional trajectory. Two distinct model goals: predictive watchlist + explanatory intervention guidance. |
| **Data Acquisition** | Pulled from live BrightHut API; joined residents, process_recordings, incident_reports, health_wellbeing_records, education_records, home_visitations, and intervention_plans. |
| **Feature Engineering** | Five behavioral domains: counseling trajectory (emotion scores, trend slope, volatility), incident history, health status trends, education engagement, and family cooperation dynamics. |
| **Exploration** | Distribution analysis, class balance review, boxplots by risk label, correlation heatmap. |
| **Explanatory Model** | Logistic regression (L2, balanced class weights). Tautological features excluded. Coefficients expressed as odds ratios for intervention design. |
| **Predictive Models** | Random Forest + Gradient Boosted Trees compared via 5-fold stratified CV. Best model selected by test AUC. Threshold tuned to maximize F2 (recall-weighted) given high cost of false negatives. |
| **Evaluation** | ROC-AUC (primary), F2-optimized threshold, confusion matrix with explicit cost framing, probability distribution by true label. |
| **Deployment** | Model artifacts saved to `model_artifacts/`. API endpoint `/api/ml/regression-risk` serves ranked at-risk list to admin/staff only. Risk badges in caseload inventory. Monthly retraining recommended. |